In [2]:
# Loading data

from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("../pdfs/report_only_text.pdf")
text_data = loader.load()
text_data

/tmp/ipykernel_1576915/1035544691.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/home/administrator/agents/pdf-question/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '../pdfs/report_only_text.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content="Here's  the  technical  version:  \nTablet  Vitals  Monitoring  System:  \n●  Automated  Vitals  Recording  Pipeline  —  Implemented  a  scheduled  monitoring  script  \nthat\n \nsamples\n \nand\n \npersists\n \ntablet\n \nvitals\n \nat\n \nconfigurable\n \nintervals\n \nfor\n \ncontinuous\n \nhardware\n \nhealth\n \ntracking\n ●  Multi-Parameter  Telemetry  Capture  —  Records  key  hardware  metrics  including  battery  \nlevel,\n \ntemperature,\n \nvoltage,\n \nand\n \ncurrent\n \ndraw\n \nfor\n \ncomprehensive\n \ndevice\n \nstate\n \nlogging\n ●  Correlation  Analysis  &  Key  Insights  —  Generates  inter-parameter  correlation  maps  \nacross\n \nrecorded\n \ntelemetry,\n \nsurfacing\n \nstatistically\n \nsignificant\n \nrelationships\n \

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap = 50
)

In [4]:
splitted_text = text_splitter.split_documents(text_data)
splitted_text

[Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '../pdfs/report_only_text.pdf', 'total_pages': 3, 'page': 0, 'page_label': '1'}, page_content="Here's  the  technical  version:  \nTablet  Vitals  Monitoring  System:  \n●  Automated  Vitals  Recording  Pipeline  —  Implemented  a  scheduled  monitoring  script  \nthat\n \nsamples\n \nand\n \npersists\n \ntablet\n \nvitals\n \nat\n \nconfigurable\n \nintervals\n \nfor\n \ncontinuous\n \nhardware\n \nhealth\n \ntracking\n ●  Multi-Parameter  Telemetry  Capture  —  Records  key  hardware  metrics  including  battery  \nlevel,\n \ntemperature,\n \nvoltage,\n \nand\n \ncurrent\n \ndraw\n \nfor\n \ncomprehensive\n \ndevice\n \nstate"),
 Document(metadata={'producer': 'Skia/PDF m150 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Untitled document', 'source': '../pdfs/report_only_text.pdf', 'total_pages': 3, 'page': 0, 

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["HF_TOKEN"] = os.getenv("HF_API_KEY") 
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") 

In [6]:
# Creating embeddings

from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22515.94it/s]


In [7]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=splitted_text,
    embedding=embedding
)

In [8]:
retriver = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)
retriver

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x70df8c378070>, search_kwargs={'k': 1})

In [9]:
retriver.invoke(
    input="statistically significant relatio"
)[0].page_content

"benchmarking\n \nand\n \nanomaly\n \ndetection\n \nHere's  the  refactored  component  with  canEdited removed  and  everything  handled  in  onChangeText:"

In [29]:
# llm model

from langchain_groq.chat_models import ChatGroq

llm = ChatGroq(
    model="qwen/qwen3-32b",    
    # reasoning_effort= "parsed"  # disable thinking
    reasoning_format="hidden",
)

In [33]:
llm_answer = llm.invoke(input="what is 2+2")
llm_answer.content

'2 + 2 equals **4**.'

In [32]:
messages = [
    (
        "system",
        "You are good at maths and doesnot like to be give long answers. Give the precise one word answer.",
    ),
    ("human", "What is radius of earth"),
]
ai_msg = llm.invoke(messages)
ai_msg.content

'6,371 km'

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage
prompt = PromptTemplate.from_template(
    [
        
    ]
)

